# core

> Fill in a module description here

In [1]:
# | default_exp gradio_app

In [2]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [1]:
from typing import Tuple
import gradio as gr
import pandas as pd
from gradio.utils import NamedString
from product_horse.api import (
    create_user_and_add_files,
    transcribe_file_and_extract_speakers,
    create_and_save_schema,
    extract_info_from_transcriptions,
    get_relevant_utterances_from_query,
    FileDict,
)
from product_horse.db import SqlModelDatabase, Transcription, Utterance
from product_horse.audio_video import AudioEditor, make_video_from_utterances
from product_horse.filesystems import LocalFileSystem
from product_horse.core import InMemoryQueue
from product_horse.extraction import AIModelClient, QADataManager, Questions
import os
from uuid import UUID
from pprint import pprint
import json
import dotenv

dotenv.load_dotenv()

# put the app working directory at ~/.product_horse/
app_working_directory = os.path.expanduser("~/.product_horse/")
if not os.path.exists(app_working_directory):
    os.makedirs(app_working_directory)
    os.makedirs(app_working_directory + "files/")


db = SqlModelDatabase(database_url="postgresql://localhost:5432/product_horse")
fs = LocalFileSystem(base_path=app_working_directory + "files/")
ai_model_client = AIModelClient()
audio_editor = AudioEditor(api_key=os.getenv("ASSEMBLYAI_API_KEY"))
queue = InMemoryQueue()


async def save_files_and_transcriptions(
    user_id: str, user_name: str, files: list[NamedString]
):
    """Save the files and transcriptions to the database."""
    if user_id is None:  # type: ignore
        raise ValueError("User ID is required")
    user_data = {"external_id": user_id, "name": user_name}
    file_dicts: list[FileDict] = []
    for file in files:
        with open(file, "rb") as f:  # Open the file in binary mode
            content = f.read()
        file_dicts.append({"content": content, "name": file.name})
    user, metadata = create_user_and_add_files(user_data, file_dicts, db, fs)
    transcriptions: list[Tuple[Transcription, list[str]]] = []
    for file_metadata in metadata:
        transcription = await transcribe_file_and_extract_speakers(
            file_metadata, db, audio_editor
        )
        transcriptions.append(transcription)
    print(user.id)
    return f"{len(transcriptions)} transcriptions saved."


def save_schema(schema_text: str):
    schema = create_and_save_schema(schema_text, db, ai_model_client)
    return schema


async def extract_info(
    user_ids: list[str],
):
    schema = db.get_latest_schema()
    if schema is None:
        raise ValueError("No schema found")
    questions = Questions(**schema.json_schema)
    transcriptions = db.get_transcriptions_by_user_ids(user_ids)
    results = await extract_info_from_transcriptions(
        schema, transcriptions, db, ai_model_client
    )

    qa_data_manager = QADataManager()

    for transcription, answers in zip(transcriptions, results):
        metadata = db.get_file_metadata(transcription.file_id)
        user_name = db.get_user(metadata.user_id).name
        if metadata is None:
            raise ValueError(f"No metadata found for file_id {transcription.file_id}")
        user_name = db.get_user(metadata.user_id).name
        qa_data_manager.add_answers(
            questions,
            answers,  # type: ignore
            document_name=str(metadata.file_name),
            user_id=str(user_name),
        )

    output_file_path = f"{app_working_directory}/output.csv"
    qa_data_manager.merge_rows_by_user()
    qa_data_manager.to_csv(file_path=output_file_path)
    return gr.File(value=output_file_path, label="Download CSV")

def get_user_names_and_transcript_counts() -> list[tuple[str, str]]:
    users = db.get_all_users()
    user_ids = [user.id for user in users]
    last_user_id = user_ids[-1]
    options: list[tuple[str, str]] = []
    for user in users:
        transcriptions = db.get_transcriptions(user.id)
        options.append(
            (f"{user.name} ({len(transcriptions)}) calls transcribed", str(user.id))
        )
    assert str(last_user_id) == options[-1][1]
    return options


def refresh_user_list() -> gr.CheckboxGroup:
    # Update the choices of the CheckboxGroup
    return gr.CheckboxGroup(
        label="User IDs",
        choices=get_user_names_and_transcript_counts(),
        interactive=True,
    )


def update_user_selection(select_all: bool) -> list[str]:
    if select_all:
        return [option[1] for option in get_user_names_and_transcript_counts()]
    else:
        return []


async def fetch_utterances(query: str, user_ids: list[str]) -> gr.Dataframe:
    transcripts = db.get_transcriptions_by_user_ids(user_ids)
    utterances = await get_relevant_utterances_from_query(query, transcripts)
    data = []
    for ut in utterances:
        duration = (ut.end - ut.start) / 1000
        selected = True if ut.speaker != "interviewer" else False
        data.append(
            [
                selected,
                ut.text,
                ut.speaker,
                duration,
                ut.start,
                ut.end,
                ut.transcription_id,
            ]
        )
    return gr.Dataframe(
        value=data,
        headers=[
            "Select",
            "Text",
            "Speaker",
            "Duration",
            "Start",
            "End",
            "Transcription ID",
        ],
        type="array",
    )


def calculate_minutes_selected(dataframe):
    dataframe = pd.DataFrame(
        dataframe,
        columns=[
            "select",
            "text",
            "speaker",
            "duration",
            "start",
            "end",
            "transcription_id",
        ],
    )
    selected_data = dataframe[dataframe["select"] == True]
    total_minutes = selected_data["duration"].sum() / 60
    return total_minutes


def create_video(dataframe, query: str) -> gr.File:
    dataframe = pd.DataFrame(
        dataframe,
        columns=[
            "select",
            "text",
            "speaker",
            "duration",
            "start",
            "end",
            "transcription_id",
        ],
    )
    selected_data = dataframe[dataframe["select"] == True].drop(
        columns=["select", "duration"]
    )
    utterances = [Utterance(**row) for index, row in selected_data.iterrows()]
    video_path = make_video_from_utterances(utterances, db, query)
    return gr.File(value=video_path, label="Download Video")


with gr.Blocks() as app:
    with gr.Tabs() as tabs:
        with gr.TabItem("File Upload and Transcription"):
            user_id_input = gr.Textbox(label="User ID")
            user_name_input = gr.Textbox(label="User Name")
            file_input = gr.File(label="Files", file_count="multiple")
            output_text = gr.Textbox(label="Output")
            save_button = gr.Button("Save Files and Transcriptions")
            save_button.click(
                save_files_and_transcriptions,
                inputs=[user_id_input, user_name_input, file_input],
                outputs=output_text,
            )
        with gr.TabItem("Edit User Files"):
            gr.Markdown("## Edit User Files")
            
            user_dropdown = gr.Dropdown(label="Select User", choices=get_user_names_and_transcript_counts())

            def fetch_files_for_user(user_id: str):
                transcriptions = db.get_transcriptions(user_id)
                files_info = [(f"{transcription.file_metadata.file_name} - {transcription.file_metadata.file_path}", transcription.file_metadata.id) for transcription in transcriptions]
                return gr.Dataframe(value=files_info, headers=["File Name and Path", "File ID"], type="array")
            
            files_display = gr.Dataframe()
            user_dropdown.change(fetch_files_for_user, inputs=user_dropdown, outputs=files_display)
            
            new_user_name_input = gr.Textbox(label="New User Name")
            update_button = gr.Button("Update User Name")
            
            def update_user_name(user_id: str, new_name: str):
                if user_id:
                    db.update_user(UUID(user_id), {"name": new_name})
                    new_choices = get_user_names_and_transcript_counts()
                    return gr.Markdown(value="User name updated successfully.", visible=True), gr.Dropdown(choices=new_choices)
                else:
                    new_choices = get_user_names_and_transcript_counts()
                    return gr.Markdown(value="User not found.", visible=True), gr.Dropdown(choices=new_choices)
                
            update_markdown = gr.Markdown(value="User name updated successfully.", visible=False)
            delete_button = gr.Button("Delete User", variant='stop')
            confirm_delete_button = gr.Button("Confirm Delete", visible=False)
            delete_confirmation_text = gr.Markdown(value="Are you sure you want to delete this user? Click 'Confirm Delete' to proceed.", visible=False)

            def prompt_delete_confirmation(user_id: str):
                if user_id:
                    return gr.Button("Confirm Delete", visible=True), gr.Textbox(value="Are you sure you want to delete this user? Click 'Confirm Delete' to proceed.", visible=True)
                else:
                    return gr.Button("Confirm Delete", visible=False), gr.Textbox(value="Are you sure you want to delete this user? Click 'Confirm Delete' to proceed.", visible=False)

            def delete_user(user_id: str):
                db.delete_user(UUID(user_id))
                new_choices = get_user_names_and_transcript_counts()
                return gr.Dropdown(choices=new_choices), gr.Button("Confirm Delete", visible=False), gr.Markdown(value="User deleted successfully.", visible=True)

            delete_button.click(prompt_delete_confirmation, inputs=[user_dropdown], outputs=[confirm_delete_button, delete_confirmation_text])
            confirm_delete_button.click(delete_user, inputs=[user_dropdown], outputs=[user_dropdown, confirm_delete_button, delete_confirmation_text])
            update_button.click(update_user_name, inputs=[user_dropdown, new_user_name_input], outputs=[update_markdown, user_dropdown])
                        
        with gr.TabItem("Schema Definition"):
            gr.Markdown("## Schema Definition")
            schema_questions_input = gr.Textbox(label="Questions")
            schema_save_button = gr.Button("Create Schema")
            schema_save_output = gr.Textbox(label="Output")
            schema_save_button.click(
                save_schema,
                inputs=[schema_questions_input],
                outputs=schema_save_output,
            )
        with gr.TabItem("Information Extraction"):
            gr.Markdown("## Information Extraction")
            select_all_checkbox = gr.Checkbox(label="Select All", value=False)
            user_ids_input = gr.CheckboxGroup(
                label="User IDs",
                choices=get_user_names_and_transcript_counts(),
                interactive=True,
            )
            refresh_button = gr.Button("Refresh")
            refresh_button.click(refresh_user_list, outputs=user_ids_input)
            select_all_checkbox.change(
                fn=update_user_selection,
                inputs=[select_all_checkbox],
                outputs=[user_ids_input],
            )
            extract_button = gr.Button("Extract Information")
            csv_download = gr.File(label="Download CSV", interactive=False)
            extract_button.click(
                extract_info, inputs=[user_ids_input], outputs=[csv_download]
            )

        with gr.TabItem("Query and Generate Video"):
            gr.Markdown("## Query and Generate Video")
            gr.Markdown(
                "Enter a query to find relevant utterances and generate a video from them. Use the transcript selector above."
            )
            query_input = gr.Textbox(label="Enter Query")
            user_ids_input_2 = gr.CheckboxGroup(
                label="User IDs",
                choices=get_user_names_and_transcript_counts(),
                interactive=True,
            )
            fetch_button = gr.Button("Fetch Utterances")
            utterance_table = gr.Dataframe()
            fetch_button.click(
                fetch_utterances,
                inputs=[query_input, user_ids_input_2],
                outputs=utterance_table,
            )

            minutes_selected = gr.Textbox(label="Minutes Selected", value=0)
            calculate_button = gr.Button("Calculate")
            calculate_button.click(
                calculate_minutes_selected,
                inputs=[utterance_table],
                outputs=minutes_selected,
            )

            create_video_button = gr.Button("Create Video")
            video_download = gr.File(label="Download Video", interactive=False)
            create_video_button.click(
                create_video,
                inputs=[utterance_table, query_input],
                outputs=[video_download],
            )

app.queue().launch()

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [4]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore